# 2D pose export on Colab (The Way Up, 22 videos)

Runs `external/pose2d/run_pose2d.py` on a Colab GPU and writes the results straight to the team Drive:
`HoldOn/data/interim/pose2d/<model>_<DATE>/<video_id>.npz` + `MANIFEST.md`.

1. **Runtime → Change runtime type → T4 GPU** (or better).
2. Edit the parameters in the next cell, then **Runtime → Run all**.
3. If Colab disconnects, reconnect and *Run all* again with the **same `DATE`**: finished videos are skipped.
4. When both runs are done, open each `MANIFEST.md` on Drive and fill in *Created by* and *Notes*.

The team folder `HoldOn` must be reachable under `MyDrive` (if it is in *Shared with me*, right-click → *Organize → Add shortcut* → My Drive).

In [ ]:
# ---- parameters ----
BRANCH = "feature/2d-line"            # repo branch to run (use "main" after the PR is merged)
DATE = "2026-11-19"                          # version date in the output folder name; keep it fixed to resume
DRIVE_HOLDON = "/content/drive/MyDrive/HoldOn"
MODELS = ["rtmw", "vitpose"]                 # run order

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf HoldOn && git clone -q --branch {BRANCH} https://github.com/zongyang078/HoldOn.git
%cd /content/HoldOn
!git log -1 --oneline
# Pose backends. Colab already ships a CUDA build of torch, numpy, scipy, pillow and opencv; keep those.
!pip3 install -q rtmlib==0.0.16 onnxruntime-gpu==1.32.0 transformers==5.23.0
# The main package (only needed for the dataset installer and the check at the end).
!pip3 install -q --no-deps -e .
import onnxruntime as ort, torch
print("onnxruntime", ort.__version__, ort.get_available_providers())
print("torch", torch.__version__, "cuda:", torch.cuda.is_available())

In [ ]:
# Copy wayup.zip from Drive (local disk is much faster than reading videos over Drive) and install it,
# verifying the SHA-256 from configs/paths.yaml.
import os, shutil
src = f"{DRIVE_HOLDON}/data/raw/wayup.zip"
assert os.path.exists(src), f"not found: {src}"
if not os.path.exists("data/raw/wayup/p1/green.mp4"):
    shutil.copy(src, "/content/wayup.zip")
    !python3 scripts/install_data.py /content/wayup.zip
!ls data/raw/wayup

In [ ]:
# Smoke test on local disk: 20 frames, both models. Must run on CUDA (the script exits otherwise).
for m in MODELS:
    !python3 external/pose2d/run_pose2d.py --model {m} --device cuda --only p1_green --max-frames 20 --out /content/smoke --overwrite

In [ ]:
# Full runs, written directly to Drive. RTMW is fast; ViTPose++-H takes most of the time.
OUT = f"{DRIVE_HOLDON}/data/interim/pose2d"
for m in MODELS:
    !python3 external/pose2d/run_pose2d.py --model {m} --device cuda --date {DATE} --out "{OUT}"

In [ ]:
# Check every exported file against the video frame counts.
import glob, numpy as np
from holdon.data.external import load_pose2d
from holdon.data.video import video_info
for folder in sorted(glob.glob(f"{OUT}/*_{DATE}")):
    files = sorted(glob.glob(f"{folder}/*.npz"))
    print(f"\n{os.path.basename(folder)}: {len(files)} files (22 expected)")
    for f in files:
        p = load_pose2d(f)
        rec, route = p.video_id.rsplit("_", 1)
        n = video_info(f"data/raw/wayup/{rec}/{route}.mp4").num_frames
        missing = int(np.isnan(p.bbox[:, 0]).sum())
        multi = int((p.num_persons > 1).sum())
        flag = "" if p.num_frames == n else f"  <-- {n} frames in video"
        print(f"  {p.video_id:12s} {p.num_frames:5d} frames, no person {missing:4d}, >1 person {multi:4d}{flag}")